# Projet B1 — Lancer l'interface Sentinelle sur Colab (GPU T4)

**Prérequis (une fois, sur Google Drive, dossier `MyDrive/b1/`)** : `b1_app/` (dossier de l'application) et `b1_bundle.zip` (produit par `export_final_bundle.ipynb`).
Runtime Colab : **GPU T4**. L'historique des conversations est écrit **sur Drive** (`MyDrive/b1/app_data/conversations.json`) : il survit à la fin de la session.

⚠️ **Sécurité** : le lien public généré à la fin donne accès à l'application à **toute personne qui l'a**, sans mot de passe (et utilise votre GPU). Ne le partagez qu'à des personnes de confiance ; arrêtez la cellule pour fermer l'accès.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, zipfile, shutil, time, re
from pathlib import Path
B1 = Path("/content/drive/MyDrive/b1")
WORK = Path("/content/b1_run")
shutil.rmtree(WORK, ignore_errors=True); shutil.copytree(B1 / "b1_app", WORK / "b1_app", ignore=shutil.ignore_patterns("data", "__pycache__", "b1_bundle"))
with zipfile.ZipFile(B1 / "b1_bundle.zip") as z: z.extractall(WORK / "b1_app")          # crée b1_app/b1_bundle/
print(sorted(p.name for p in (WORK / "b1_app").iterdir()))

In [ ]:
# !pip install -q -r /content/b1_run/b1_app/requirements.txt

## Démarrage du serveur Streamlit puis du tunnel public
Le **premier chargement** de la page télécharge Qwen2.5-7B (≈ 15 Go) puis le quantifie : compter 3 à 8 minutes. Les ouvertures suivantes sont immédiates tant que la session tourne.

In [ ]:
env = dict(os.environ, B1_DATA_DIR=str(B1 / "app_data"), B1_BUNDLE_DIR=str(WORK / "b1_app" / "b1_bundle"))
server = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                          cwd=WORK / "b1_app", env=env, stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8); print(open("/content/streamlit.log").read()[-600:])

In [ ]:
if not Path("/content/cloudflared").exists():
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
print("Interface :", url or "URL non trouvée — relancer la cellule")

Laisser la cellule précédente et la session ouvertes tant que l'interface est utilisée. Pour arrêter : exécuter la cellule suivante.

In [ ]:
tun.terminate(); server.terminate(); print("arrêté")